# Tool Calling Debug Notebook

Interactive notebook for testing tool-calling request structures against `llama-server`.

Use this to:
- Start a local `llama-server` instance programmatically
- Send requests via the raw `openai` client
- Inspect request/response payloads to debug why a model does or does not emit `tool_calls`

**Workflow:** Edit Cell 2 (config), run cells top-to-bottom, inspect results.

In [1]:
import json
import os
import subprocess
import time
from pathlib import Path

import requests
from openai import OpenAI

## 2. Configuration

Edit the variables below for your model and desired server settings.

In [2]:
# --- EDIT THESE ---
MODEL_PATH = "../models/gemma-4-E2B-it-IQ4_NL.gguf"   # path to GGUF file
PORT = 8082
CHAT_TEMPLATE = None                        # e.g. "gemma", "chatml", "llama3"
CHAT_TEMPLATE_FILE = None                   # e.g. "../gemma-3-tool-template.jinja"
CONTEXT_SIZE = 4096
SERVER_BINARY = None                        # override if llama-server is not on PATH

# --- DERIVED ---
BASE_URL = f"http://localhost:{PORT}"
MODEL_NAME = Path(MODEL_PATH).stem

## 3. Server Lifecycle Helpers

In [3]:
def resolve_binary(explicit: str | None = None) -> str:
    """Find llama-server binary."""
    import shutil
    candidates = []
    if explicit:
        candidates.append(explicit)
    candidates.extend([
        "llama-server",
        "/opt/homebrew/bin/llama-server",
        "/usr/local/bin/llama-server",
        "../build/bin/llama-server",
        "../../build/bin/llama-server",
    ])
    for path in candidates:
        resolved = shutil.which(path) if not os.path.isabs(path) else path
        if resolved and os.path.isfile(resolved) and os.access(resolved, os.X_OK):
            return resolved
    raise RuntimeError(f"Could not find llama-server. Tried: {candidates}")


def start_server(model_path: str, port: int, chat_template=None, chat_template_file=None,
                 context_size=4096, server_binary=None):
    """Start llama-server and return the subprocess.Popen object."""
    binary = resolve_binary(server_binary)
    cmd = [
        binary,
        "--jinja",
        "-m", model_path,
        "--port", str(port),
        "-c", str(context_size),
    ]
    if chat_template:
        cmd.extend(["--chat-template", chat_template])
    if chat_template_file:
        cmd.extend(["--chat-template-file", chat_template_file])

    print(f"[server] Starting: {' '.join(cmd)}")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    # Poll /v1/models until ready
    for attempt in range(60):
        try:
            r = requests.get(f"http://localhost:{port}/v1/models", timeout=2)
            if r.status_code == 200:
                print(f"[server] Ready on port {port}")
                return proc
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(1)
    proc.terminate()
    raise RuntimeError("Server did not become ready in 60s")


def stop_server(proc: subprocess.Popen):
    """Gracefully stop the server."""
    if proc is None:
        return
    print("[server] Stopping...")
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait()
    print("[server] Stopped.")


def get_props(port: int) -> dict:
    """Fetch /props from the running server."""
    r = requests.get(f"http://localhost:{port}/props", timeout=5)
    r.raise_for_status()
    return r.json()


def pretty_print(title: str, obj: dict):
    """Pretty-print a dict with a header."""
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(json.dumps(obj, indent=2, default=str))

## 4. Start Server

In [4]:
proc = start_server(
    model_path=MODEL_PATH,
    port=PORT,
    chat_template=CHAT_TEMPLATE,
    chat_template_file=CHAT_TEMPLATE_FILE,
    context_size=CONTEXT_SIZE,
    server_binary=SERVER_BINARY,
)

[server] Starting: /opt/homebrew/bin/llama-server --jinja -m ../models/gemma-4-E2B-it-IQ4_NL.gguf --port 8082 -c 4096
[server] Ready on port 8082


## 5. Inspect Server Props

Check `chat_template_caps.supports_tools` — if this is `false`, the model will never emit native `tool_calls` regardless of request format.

In [5]:
props = get_props(PORT)
pretty_print("Server Props", props)

caps = props.get("chat_template_caps", {})
supports_tools = caps.get("supports_tools")
print(f"\n>>> supports_tools = {supports_tools}")
if supports_tools is False:
    print("WARNING: Server reports supports_tools=false. Tool calling may not work.")


  Server Props
{
  "default_generation_settings": {
    "params": {
      "seed": 4294967295,
      "temperature": 1.0,
      "dynatemp_range": 0.0,
      "dynatemp_exponent": 1.0,
      "top_k": 64,
      "top_p": 0.949999988079071,
      "min_p": 0.05000000074505806,
      "top_n_sigma": -1.0,
      "xtc_probability": 0.0,
      "xtc_threshold": 0.10000000149011612,
      "typical_p": 1.0,
      "repeat_last_n": 64,
      "repeat_penalty": 1.0,
      "presence_penalty": 0.0,
      "frequency_penalty": 0.0,
      "dry_multiplier": 0.0,
      "dry_base": 1.75,
      "dry_allowed_length": 2,
      "dry_penalty_last_n": -1,
      "mirostat": 0,
      "mirostat_tau": 5.0,
      "mirostat_eta": 0.10000000149011612,
      "max_tokens": -1,
      "n_predict": -1,
      "n_keep": 0,
      "n_discard": 0,
      "ignore_eos": false,
      "stream": true,
      "n_probs": 0,
      "min_keep": 0,
      "chat_format": "Content-only",
      "reasoning_format": "none",
      "reasoning_in_content":

## 6. Initialize OpenAI Client

In [6]:
client = OpenAI(base_url=f"{BASE_URL}/v1", api_key="not-needed")

## 7. Sanity Check (No Tools)

Verify the model responds to a plain chat completion.

In [7]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Say hello"}],
    temperature=0.0,
)
pretty_print("Sanity Check Response", response.model_dump())
print("\n>>> Content:", response.choices[0].message.content)


  Sanity Check Response
{
  "id": "chatcmpl-mMGLPimZM5Xmnhh79z3DqCxPBsJYbPMJ",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Hello!",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1780161018,
  "model": "gemma-4-E2B-it-IQ4_NL.gguf",
  "object": "chat.completion",
  "service_tier": null,
  "system_fingerprint": "b9260-3a6db741a",
  "usage": {
    "completion_tokens": 3,
    "prompt_tokens": 18,
    "total_tokens": 21,
    "completion_tokens_details": null,
    "prompt_tokens_details": {
      "audio_tokens": null,
      "cached_tokens": 0
    }
  },
  "timings": {
    "cache_n": 0,
    "prompt_n": 18,
    "prompt_ms": 621.448,
    "prompt_per_token_ms": 34.52488888888889,
    "prompt_per_second": 28.964611681106064,
    "predicted_n": 3,
    "predicted_

## 8. Standard Tool Calling

Send the exact OpenAI-format `tools` array and check if `tool_calls` is populated.

In [8]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "The city name"},
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit",
                    },
                },
                "required": ["city"],
            },
        },
    }
]

messages = [
    {"role": "system", "content": "You are a helpful assistant with access to tools."},
    {"role": "user", "content": "What is the weather like in Tokyo in celsius?"},
]

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    tools=tools,
    tool_choice="auto",
    temperature=0.0,
)

pretty_print("Standard Tool Call Response", response.model_dump())

msg = response.choices[0].message
print(f"\n>>> Has tool_calls: {msg.tool_calls is not None and len(msg.tool_calls) > 0}")
if msg.tool_calls:
    for tc in msg.tool_calls:
        print(f"    Tool: {tc.function.name}")
        print(f"    Args: {tc.function.arguments}")


  Standard Tool Call Response
{
  "id": "chatcmpl-iUCwyfjDqJUGa6QSab6dJx4Hj5LcYzxS",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "1PnLc75VTv4KLxmGgdpKuMLl1yGgzziq",
            "function": {
              "arguments": "{\"city\":\"Tokyo\",\"unit\":\"celsius\"}",
              "name": "get_weather"
            },
            "type": "function"
          }
        ],
        "reasoning_content": "1.  **Analyze the user's request:** The user is asking for the weather in \"Tokyo\" and specifically wants the temperature in \"celsius\".\n\n2.  **Examine available tools:** The available tool is `get_weather(city: STRING, unit: STRING)`.\n\n3.  **Determine the necessary arguments for the tool:**\n    *   `cit

## 9. Gemma Debugging Matrix

Test multiple combinations to isolate what makes a model emit tool calls.

Combinations tested:
- `tool_choice`: `auto`, `required`, `none`
- System message: present vs absent
- Tool delivery: native `tools` param vs manual text injection

In [ ]:
def test_combo(tool_choice, with_system=True, manual_tools=False):
    """Run one combination and return whether tool_calls were emitted."""
    user_msg = "What is the weather like in Tokyo in celsius?"
    msgs = []
    if with_system:
        msgs.append({"role": "system", "content": "You are a helpful assistant with access to tools."})
    if manual_tools:
        tool_text = "\n\nAvailable tools:\n- get_weather: Get the current weather for a given location.\n  Parameters: {\"city\": string, \"unit\": \"celsius\"|\"fahrenheit\"}\n\nWhen you need to use a tool, respond with: <call>name: get_weather\narguments: <json></call>\n"
        user_msg += tool_text
        req_tools = None
    else:
        req_tools = tools
    msgs.append({"role": "user", "content": user_msg})

    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=msgs,
        tools=req_tools,
        tool_choice=tool_choice,
        temperature=0.0,
    )
    msg = resp.choices[0].message
    has_tools = msg.tool_calls is not None and len(msg.tool_calls) > 0
    return {
        "tool_choice": tool_choice,
        "with_system": with_system,
        "manual_tools": manual_tools,
        "has_tool_calls": has_tools,
        "content_preview": (msg.content or "")[:200],
    }


results = []
for tc in ["auto", "required", "none"]:
    for sys in [True, False]:
        for manual in [False, True]:
            results.append(test_combo(tc, with_system=sys, manual_tools=manual))

print(f"{'tool_choice':<12} {'system':<8} {'manual':<8} {'tool_calls?':<12} {'content_preview'}")
print("-" * 80)
for r in results:
    print(f"{r['tool_choice']:<12} {str(r['with_system']):<8} {str(r['manual_tools']):<8} {str(r['has_tool_calls']):<12} {r['content_preview'][:40]}")

## 10. Request/Response Inspector

Send any arbitrary request and inspect the exact payload and raw response.

In [ ]:
def inspect_request(messages, tools=None, **kwargs):
    """Send a request and pretty-print both sides."""
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": kwargs.get("temperature", 0.0),
    }
    if tools:
        payload["tools"] = tools
        payload["tool_choice"] = kwargs.get("tool_choice", "auto")
    for k, v in kwargs.items():
        payload[k] = v

    pretty_print("REQUEST PAYLOAD", payload)

    resp = client.chat.completions.create(**payload)
    pretty_print("RAW RESPONSE", resp.model_dump())

    msg = resp.choices[0].message
    print(f"\n>>> Content: {msg.content}")
    print(f">>> Tool calls: {msg.tool_calls}")
    return resp


# Example usage — edit and re-run this cell as needed
inspect_request(
    messages=[
        {"role": "user", "content": "Calculate 2+2"},
    ],
    tools=[
        {
            "type": "function",
            "function": {
                "name": "calculator",
                "description": "Evaluate a math expression.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "expression": {"type": "string"}
                    },
                    "required": ["expression"],
                },
            },
        }
    ],
)

## 11. Cleanup

Stop the server when you're done.

In [ ]:
stop_server(proc)